## 04 strudy DDF

Cumulative 5-sigma depth versus time for Rubin LSST Deep Drilling Fields.

    Input:
        An OpSim SQLite database containing observations with
        observationStartMJD, filter, fiveSigmaDepth and scheduler_note/note.

    Output:
        ddf_cumulative_depth.png
        ddf_cumulative_depth.pdf

In [ ]:
import re
import sqlite3
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

## Configuration

In [ ]:
# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SURVEY_START = date(2026, 6, 29)  # Change if needed
N_YEARS = 10

# Standard LSST filter colours
BAND_COLORS = {
    "u": "#4477AA",
    "g": "#228833",
    "r": "#CC3311",
    "i": "#EE7733",
    "z": "#AA3377",
    "y": "#66CCEE",
}

BANDS = ["u", "g", "r", "i", "z", "y"]

DDF_NAMES = [
    "COSMOS",
    "ECDFS",
    "EDFS",
    "ELAIS-S1",
    "XMM-LSS",
]

## Utility Functions

In [ ]:
# ------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------


def mjd_to_datetime(mjd):
    """Convert Modified Julian Date to Python datetime."""
    epoch = datetime(1858, 11, 17)
    return epoch + timedelta(days=float(mjd))


def survey_year_date(n):
    """Calendar date marking the beginning of survey year n+1."""
    try:
        return SURVEY_START.replace(year=SURVEY_START.year + n)
    except ValueError:
        # Handle 29 February if SURVEY_START is changed.
        return SURVEY_START.replace(year=SURVEY_START.year + n, day=28)


def identify_ddf(text):
    """Identify a DDF from scheduler_note or note."""
    if not isinstance(text, str):
        return None

    s = text.upper().replace("_", "-")

    # Match names while allowing common OpSim spellings.
    aliases = {
        "COSMOS": r"COSMOS",
        "ECDFS": r"ECDFS",
        "EDFS": r"EDFS",
        "ELAIS-S1": r"ELAIS[- ]?S?1|ELAISS1",
        "XMM-LSS": r"XMM[- ]?LSS|XMMLSS",
    }

    # Require a DD indication to avoid accidentally selecting
    # an unrelated field with a similar name.
    if not re.search(r"\bDD\b|DEEP.?DRILL", s):
        return None

    for name, pattern in aliases.items():
        if re.search(pattern, s):
            return name

    return None

# Read OpSim database

In [ ]:
def load_ddf_visits(db_path):
    with sqlite3.connect(db_path) as con:
        tables = pd.read_sql_query("SELECT name FROM sqlite_master " "WHERE type='table'", con)[
            "name"
        ].tolist()

        if "observations" not in tables:
            raise RuntimeError("Table 'observations' not found. " f"Tables available: {tables}")

        columns = pd.read_sql_query("PRAGMA table_info(observations)", con)["name"].tolist()

        required = ["observationStartMJD", "filter", "fiveSigmaDepth"]
        missing = [c for c in required if c not in columns]
        if missing:
            raise RuntimeError(f"Missing columns: {missing}\n" f"Available columns: {columns}")

        # Different OpSim versions may use different note columns.
        note_cols = [c for c in ("scheduler_note", "note", "fieldName") if c in columns]

        selected = required + [c for c in note_cols if c not in required]

        sql = "SELECT " + ", ".join(f'"{c}"' for c in selected) + " FROM observations"

        df = pd.read_sql_query(sql, con)

    if not note_cols:
        raise RuntimeError("No scheduler_note/note/fieldName column found.")

    # Combine note fields to make the identification more robust.
    df["ddf"] = None
    for col in note_cols:
        identified = df[col].apply(identify_ddf)
        df["ddf"] = df["ddf"].fillna(identified)

    df = df[df["ddf"].notna()].copy()

    if df.empty:
        raise RuntimeError(
            "No DDF visits identified. Inspect the values of "
            "scheduler_note and note in your OpSim database."
        )

    df["observationStartMJD"] = pd.to_numeric(df["observationStartMJD"], errors="coerce")
    df["fiveSigmaDepth"] = pd.to_numeric(df["fiveSigmaDepth"], errors="coerce")
    df = df.dropna(subset=["observationStartMJD", "fiveSigmaDepth"])
    df = df[df["filter"].isin(BANDS)]
    df = df.sort_values("observationStartMJD")

    return df

## Cumulative depth

In [ ]:
def add_cumulative_depth(df):
    """
    Combine independent 5-sigma visits in each filter.

    m5_cum = 1.25 log10(sum_i 10**(0.8*m5_i))
    """
    df = df.sort_values("observationStartMJD").copy()

    # Relative weights avoid unnecessary numerical overflow.
    m5_max = df["fiveSigmaDepth"].max()
    weights = 10.0 ** (0.8 * (df["fiveSigmaDepth"].to_numpy() - m5_max))

    df["weight"] = weights
    df["weight_cum"] = np.cumsum(weights)
    df["m5_cum"] = m5_max + 1.25 * np.log10(df["weight_cum"])

    df["date"] = df["observationStartMJD"].apply(mjd_to_datetime)

    return df

## Plot

In [ ]:
# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------


def plot_ddfs(df, output="ddf_cumulative_depth"):
    ddfs = sorted(df["ddf"].unique())

    fig, axes = plt.subplots(
        len(ddfs),
        1,
        figsize=(15, 4.5 * len(ddfs)),
        sharex=True,
        constrained_layout=True,
    )

    if len(ddfs) == 1:
        axes = [axes]

    for ax, ddf in zip(axes, ddfs):
        subset = df[df["ddf"] == ddf]

        for band in BANDS:
            visits = subset[subset["filter"] == band]
            if visits.empty:
                continue

            visits = add_cumulative_depth(visits)

            ax.plot(
                visits["date"],
                visits["m5_cum"],
                color=BAND_COLORS[band],
                label=band,
                linewidth=1.8,
            )

        # Mark the beginning of each survey year.
        for n in range(N_YEARS + 1):
            boundary = survey_year_date(n)

            ax.axvline(
                boundary,
                color="0.55",
                linestyle="--",
                linewidth=0.8,
                alpha=0.7,
            )

            if n < N_YEARS:
                midpoint = boundary + (survey_year_date(n + 1) - boundary) / 2

                ax.text(
                    midpoint,
                    0.97,
                    f"Y{n + 1}",
                    transform=ax.get_xaxis_transform(),
                    ha="center",
                    va="top",
                    fontsize=9,
                    color="0.3",
                )

        ax.set_title(ddf, fontsize=15, loc="left")
        ax.set_ylabel(r"Cumulative $m_5$ (mag)", fontsize=12)
        ax.grid(True, alpha=0.25)
        ax.legend(
            title="LSST band",
            ncol=6,
            loc="lower right",
            frameon=True,
        )

    axes[-1].set_xlabel("Date", fontsize=12)
    axes[-1].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    fig.suptitle(
        "Rubin LSST — Cumulative DDF Depth by Filter",
        fontsize=17,
    )

    start = datetime.combine(SURVEY_START, datetime.min.time())
    end = datetime.combine(survey_year_date(N_YEARS), datetime.min.time())
    axes[-1].set_xlim(start, end)

    fig.savefig(f"{output}.png", dpi=180, bbox_inches="tight")
    fig.savefig(f"{output}.pdf", bbox_inches="tight")
    plt.show()

    print(f"Saved: {output}.png")
    print(f"Saved: {output}.pdf")

# Main

In [ ]:
opsdb = "/Users/dagoret/DATA/OpSim/baseline_v5.3.6_10yrs.db"

In [ ]:
df = load_ddf_visits(opsdb)

In [ ]:
df

In [ ]:
print("\nDDF visits identified:")
print(df.groupby("ddf").size().to_string())

print("\nVisit counts by DDF and filter:")
print(df.groupby(["ddf", "filter"]).size().unstack(fill_value=0).to_string())